In [ ]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
from scipy import stats
import statsmodels.api as sm
from statsmodels.formula.api import ols, logit

In [ ]:
df_full = pd.read_csv("flight_data.csv")
df = df_full.sample(n=10_000, random_state=12310048).reset_index(drop=True)

In [ ]:
print("\n" + "─" * 70)
print("Q1. Coach Ticket Price Overview")
print("─" * 70)

cp = df["coach_price"]

q1_stats = {
    "Min"    : cp.min(),
    "Max"    : cp.max(),
    "Mean"   : cp.mean(),
    "Median" : cp.median(),
    "Std Dev": cp.std(),
    "Q1"     : cp.quantile(0.25),
    "Q3"     : cp.quantile(0.75),
}
for k, v in q1_stats.items():
    print(f"  {k:<10}: ${v:,.2f}")

pct_above_500 = (cp > 500).mean() * 100
print(f"\n  Flights above $500 : {pct_above_500:.1f}%")
print(f"\n  Interpretation:")
print(f"  The average coach price is ${cp.mean():.2f} with a median of ${cp.median():.2f}.")
print(f"  Only {pct_above_500:.1f}% of tickets cost more than $500.")
print(f"  $500 is above average — it would be considered an expensive coach ticket.")

# Plot Q1
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].hist(cp, bins=40, color="steelblue", edgecolor="white")
axes[0].axvline(cp.mean(),   color="red",    linestyle="--", label=f"Mean  ${cp.mean():.0f}")
axes[0].axvline(cp.median(), color="orange", linestyle="--", label=f"Median ${cp.median():.0f}")
axes[0].axvline(500,         color="green",  linestyle="--", label="$500 mark")
axes[0].set_title("Q1 – Histogram of Coach Prices")
axes[0].set_xlabel("Coach Price ($)")
axes[0].set_ylabel("Count")
axes[0].legend()

axes[1].boxplot(cp, vert=True, patch_artist=True,
                boxprops=dict(facecolor="steelblue", color="navy"),
                medianprops=dict(color="orange", linewidth=2))
axes[1].axhline(500, color="green", linestyle="--", label="$500 mark")
axes[1].set_title("Q1 – Boxplot of Coach Prices")
axes[1].set_ylabel("Coach Price ($)")
axes[1].legend()
plt.tight_layout()
plt.savefig("q1_coach_price.png")
plt.show()

In [ ]:
cp = df['coach_price']

cp.describe()
 
# % above $500
pct = (cp > 500).mean() * 100
print(f'Above $500: {pct:.1f}%')
 
# Visualisation
import matplotlib.pyplot as plt
plt.hist(cp, bins=40, color='steelblue')
plt.axvline(cp.mean(), color='red',
            linestyle='--', label='Mean')
plt.axvline(500, color='green',
            linestyle='--', label='$500')
plt.legend(); plt.show()


In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# Q2. High, Low, Average for 8-Hour Flights; is $500 reasonable?
# ══════════════════════════════════════════════════════════════════════════════
print("\n" + "─" * 70)
print("Q2. Coach Prices for 8-Hour Flights")
print("─" * 70)

df8 = df[df["hours"] == 8]["coach_price"]
print(f"  Number of 8-hour flights in sample : {len(df8)}")

q2_stats = {
    "Min"   : df8.min(),
    "Max"   : df8.max(),
    "Mean"  : df8.mean(),
    "Median": df8.median(),
    "Std"   : df8.std(),
}
for k, v in q2_stats.items():
    print(f"  {k:<10}: ${v:,.2f}")

pct8_above_500 = (df8 > 500).mean() * 100
print(f"\n  8-hr flights above $500 : {pct8_above_500:.1f}%")
print(f"\n  Interpretation:")
print(f"  For 8-hour flights, the average price is ${df8.mean():.2f}.")
print(f"  {pct8_above_500:.1f}% cost over $500, making $500 more reasonable for long-haul.")

# Plot Q2
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].hist(df8, bins=20, color="darkorange", edgecolor="white")
axes[0].axvline(df8.mean(), color="red",   linestyle="--", label=f"Mean ${df8.mean():.0f}")
axes[0].axvline(500,        color="green", linestyle="--", label="$500 mark")
axes[0].set_title("Q2 – Coach Prices: 8-Hour Flights")
axes[0].set_xlabel("Coach Price ($)")
axes[0].legend()

axes[1].boxplot(df8, patch_artist=True,
                boxprops=dict(facecolor="darkorange"),
                medianprops=dict(color="darkred", linewidth=2))
axes[1].axhline(500, color="green", linestyle="--", label="$500 mark")
axes[1].set_title("Q2 – Boxplot: 8-Hour Flights")
axes[1].set_ylabel("Coach Price ($)")
axes[1].legend()
plt.tight_layout()
plt.savefig("q2_8hour_prices.png")
plt.show()

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# Q3. Flight Delay Distribution — typical delays, significant delays
# ══════════════════════════════════════════════════════════════════════════════
print("\n" + "─" * 70)
print("Q3. Flight Delay Distribution")
print("─" * 70)

delay = df["delay"]
print(f"  Mean delay   : {delay.mean():.2f} min")
print(f"  Median delay : {delay.median():.2f} min")
print(f"  Std Dev      : {delay.std():.2f} min")
print(f"  Max delay    : {delay.max():.0f} min")
print(f"  No delay (0) : {(delay == 0).mean()*100:.1f}%")
print(f"  > 15 min     : {(delay > 15).mean()*100:.1f}%  (significant connection risk)")
print(f"  > 60 min     : {(delay > 60).mean()*100:.1f}%  (severe delay)")
print(f"\n  Interpretation:")
print(f"  Most flights have zero or minimal delay. About {(delay>15).mean()*100:.1f}% have")
print(f"  delays over 15 minutes which could affect connecting flights.")

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].hist(delay, bins=40, color="tomato", edgecolor="white")
axes[0].axvline(15, color="orange", linestyle="--", label="15 min (connection risk)")
axes[0].axvline(60, color="darkred", linestyle="--", label="60 min (severe)")
axes[0].set_title("Q3 – Histogram of Delay Times")
axes[0].set_xlabel("Delay (minutes)")
axes[0].set_ylabel("Count")
axes[0].legend()

axes[1].boxplot(delay, patch_artist=True,
                boxprops=dict(facecolor="tomato"),
                medianprops=dict(color="darkred", linewidth=2))
axes[1].set_title("Q3 – Boxplot of Delay Times")
axes[1].set_ylabel("Delay (minutes)")
plt.tight_layout()
plt.savefig("q3_delays.png")
plt.show()

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# Q4. Correlation: Coach Price vs Miles, Passengers, Delay, Hours
# ══════════════════════════════════════════════════════════════════════════════
print("\n" + "─" * 70)
print("Q4. Correlation: Coach Price vs Numeric Predictors")
print("─" * 70)

num_vars = ["miles", "passengers", "delay", "hours"]
for v in num_vars:
    r, p = stats.pearsonr(df[v], df["coach_price"])
    sig = "✓ significant" if p < 0.05 else "✗ not significant"
    print(f"  coach_price ~ {v:<12}: r = {r:+.4f}  p = {p:.4f}  {sig}")

print(f"\n  Interpretation:")
print(f"  Miles and hours are the strongest positive predictors of coach price.")
print(f"  Passengers and delay have weaker/negligible correlations.")

# Scatter grid
fig, axes = plt.subplots(2, 2, figsize=(12, 8))
colors = ["steelblue", "darkorange", "tomato", "seagreen"]
for ax, var, col in zip(axes.flat, num_vars, colors):
    ax.scatter(df[var], df["coach_price"], alpha=0.15, s=5, color=col)
    m, b = np.polyfit(df[var], df["coach_price"], 1)
    x_line = np.linspace(df[var].min(), df[var].max(), 100)
    ax.plot(x_line, m*x_line + b, color="black", linewidth=1.5)
    r, _ = stats.pearsonr(df[var], df["coach_price"])
    ax.set_title(f"Coach Price vs {var.capitalize()} (r={r:+.3f})")
    ax.set_xlabel(var)
    ax.set_ylabel("Coach Price ($)")
plt.suptitle("Q4 – Correlation Scatterplots", fontsize=14, y=1.01)
plt.tight_layout()
plt.savefig("q4_correlations.png")
plt.show()

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# Q5. Relationship: Coach Price vs First-Class Price
# ══════════════════════════════════════════════════════════════════════════════
print("\n" + "─" * 70)
print("Q5. Coach Price vs First-Class Price")
print("─" * 70)

r5, p5 = stats.pearsonr(df["coach_price"], df["firstclass_price"])
print(f"  Pearson r = {r5:.4f},  p-value = {p5:.4e}")
ratio = df["firstclass_price"] / df["coach_price"]
print(f"  Avg first-class / coach ratio : {ratio.mean():.2f}x")
print(f"\n  Interpretation:")
print(f"  Very strong positive correlation (r={r5:.3f}). Flights with higher")
print(f"  coach prices almost always have higher first-class prices too.")
print(f"  First class is typically {ratio.mean():.1f}x the coach price.")

fig, ax = plt.subplots(figsize=(7, 5))
ax.scatter(df["coach_price"], df["firstclass_price"], alpha=0.15, s=5, color="purple")
m, b = np.polyfit(df["coach_price"], df["firstclass_price"], 1)
x_l = np.linspace(df["coach_price"].min(), df["coach_price"].max(), 100)
ax.plot(x_l, m * x_l + b, color="black", linewidth=1.5)
ax.set_title(f"Q5 – Coach vs First-Class Price (r = {r5:.3f})")
ax.set_xlabel("Coach Price ($)")
ax.set_ylabel("First-Class Price ($)")
plt.tight_layout()
plt.savefig("q5_coach_vs_first.png")
plt.show()

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# Q6. Coach Price vs In-Flight Features (meal, entertainment, wifi)
# ══════════════════════════════════════════════════════════════════════════════
print("\n" + "─" * 70)
print("Q6. Coach Price vs In-Flight Features")
print("─" * 70)

features = ["inflight_meal", "inflight_entertainment", "inflight_wifi"]
for feat in features:
    g_yes = df[df[feat] == "Yes"]["coach_price"]
    g_no = df[df[feat] == "No"]["coach_price"]
    diff = g_yes.mean() - g_no.mean()
    t, p = stats.ttest_ind(g_yes, g_no)
    sig = "✓ significant" if p < 0.05 else "✗ not significant"
    print(f"  {feat}")
    print(
        f"    Yes: ${g_yes.mean():.2f}  |  No: ${g_no.mean():.2f}  |  Diff: ${diff:+.2f}"
    )
    print(f"    t = {t:.3f}, p = {p:.4f}  {sig}")

print(f"\n  Interpretation:")
print(f"  All three features are associated with higher coach prices.")
print(f"  Entertainment tends to have the largest price premium.")

fig, ax = plt.subplots(figsize=(9, 5))
data_plot = []
labels = []
for feat in features:
    for val in ["Yes", "No"]:
        data_plot.append(df[df[feat] == val]["coach_price"].values)
        labels.append(f"{feat.split('_')[-1].title()}\n({val})")

bp = ax.boxplot(
    data_plot,
    labels=labels,
    patch_artist=True,
    medianprops=dict(color="black", linewidth=2),
)
colors_box = ["#4CAF50", "#F44336"] * 3
for patch, col in zip(bp["boxes"], colors_box):
    patch.set_facecolor(col)
    patch.set_alpha(0.7)
ax.set_title("Q6 – Coach Price by In-Flight Features")
ax.set_ylabel("Coach Price ($)")
plt.tight_layout()
plt.savefig("q6_features.png")
plt.show()

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# Q7. Passengers vs Flight Duration (Hours)
# ══════════════════════════════════════════════════════════════════════════════
print("\n" + "─" * 70)
print("Q7. Passengers vs Flight Duration")
print("─" * 70)

r7, p7 = stats.pearsonr(df["hours"], df["passengers"])
avg_by_hour = df.groupby("hours")["passengers"].mean()
print(f"  Pearson r = {r7:.4f},  p-value = {p7:.4f}")
print("\n  Average passengers by flight hours:")
print(avg_by_hour.to_string())
print(f"\n  Interpretation:")
print(f"  Correlation r={r7:.3f}. Longer flights tend to carry more passengers,")
print(f"  likely due to larger aircraft used on longer routes.")

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].scatter(df["hours"], df["passengers"], alpha=0.15, s=5, color="teal")
axes[0].set_title(f"Q7 – Passengers vs Hours (r={r7:.3f})")
axes[0].set_xlabel("Hours")
axes[0].set_ylabel("Passengers")

avg_by_hour.plot(kind="bar", ax=axes[1], color="teal", edgecolor="white")
axes[1].set_title("Q7 – Avg Passengers by Flight Duration")
axes[1].set_xlabel("Hours")
axes[1].set_ylabel("Avg Passengers")
plt.tight_layout()
plt.savefig("q7_passengers_hours.png")
plt.show()

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# Q8. Coach & First-Class Prices: Weekend vs Weekday
# ══════════════════════════════════════════════════════════════════════════════
print("\n" + "─" * 70)
print("Q8. Coach & First-Class Prices: Weekend vs Weekday")
print("─" * 70)

for price_col in ["coach_price", "firstclass_price"]:
    wkend = df[df["weekend"] == "Yes"][price_col]
    wkday = df[df["weekend"] == "No"][price_col]
    t, p = stats.ttest_ind(wkend, wkday)
    print(f"  {price_col}:")
    print(
        f"    Weekend avg : ${wkend.mean():.2f}  |  Weekday avg : ${wkday.mean():.2f}"
    )
    print(f"    t = {t:.3f}, p = {p:.4f}  {'✓ sig' if p < 0.05 else '✗ not sig'}")

print(f"\n  Interpretation:")
print(f"  Weekend and weekday prices may differ slightly — t-test determines")
print(f"  whether the difference is statistically meaningful.")

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for ax, col, color in zip(
    axes, ["coach_price", "firstclass_price"], ["steelblue", "mediumpurple"]
):
    data8 = [df[df["weekend"] == v][col].values for v in ["Yes", "No"]]
    bp = ax.boxplot(
        data8,
        labels=["Weekend", "Weekday"],
        patch_artist=True,
        medianprops=dict(color="black", linewidth=2),
    )
    for patch in bp["boxes"]:
        patch.set_facecolor(color)
        patch.set_alpha(0.7)
    ax.set_title(f"Q8 – {col.replace('_', ' ').title()}: Weekend vs Weekday")
    ax.set_ylabel("Price ($)")
plt.tight_layout()
plt.savefig("q8_weekend_weekday.png")
plt.show()

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# Q9. Coach Prices: Redeye vs Non-Redeye by Day of Week
# ══════════════════════════════════════════════════════════════════════════════
print("\n" + "─" * 70)
print("Q9. Coach Prices: Redeye vs Non-Redeye by Day of Week")
print("─" * 70)

day_order = [
    "Monday",
    "Tuesday",
    "Wednesday",
    "Thursday",
    "Friday",
    "Saturday",
    "Sunday",
]
pivot = df.pivot_table(
    values="coach_price", index="day_of_week", columns="redeye", aggfunc="mean"
)
pivot = pivot.reindex(day_order)
print(pivot.round(2))

print(f"\n  Interpretation:")
print(f"  Redeye flights generally differ in price by day. Comparing across days")
print(f"  reveals which combination of day + redeye commands a premium.")

pivot.plot(
    kind="bar", figsize=(10, 5), color=["steelblue", "tomato"], edgecolor="white"
)
plt.title("Q9 – Coach Price: Redeye vs Non-Redeye by Day")
plt.xlabel("Day of Week")
plt.ylabel("Avg Coach Price ($)")
plt.legend(title="Redeye", labels=["No", "Yes"])
plt.xticks(rotation=30)
plt.tight_layout()
plt.savefig("q9_redeye_by_day.png")
plt.show()

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# Q10. Comprehensive Statistical Analysis
# ══════════════════════════════════════════════════════════════════════════════
print("\n" + "═" * 70)
print("Q10. COMPREHENSIVE STATISTICAL ANALYSIS")
print("═" * 70)

# ── 10a. Summary Statistics ────────────────────────────────────────────────
print("\n── 10a. Summary Statistics ──")
numeric_cols = [
    "miles",
    "passengers",
    "delay",
    "coach_price",
    "firstclass_price",
    "hours",
]
summary = df[numeric_cols].agg(["mean", "median", "std", "min", "max"]).round(3)
print(summary.to_string())

# ── 10b. Visualizations ───────────────────────────────────────────────────
print("\n── 10b. Visualizations (Histograms, Boxplots, Bar Charts) ──")

# Histograms
fig, axes = plt.subplots(2, 3, figsize=(15, 8))
for ax, col in zip(axes.flat, numeric_cols):
    ax.hist(df[col], bins=30, color="steelblue", edgecolor="white")
    ax.set_title(f"Histogram – {col}")
    ax.set_xlabel(col)
    ax.set_ylabel("Count")
plt.suptitle("Q10b – Histograms of Numeric Variables", fontsize=14)
plt.tight_layout()
plt.savefig("q10b_histograms.png")
plt.show()

# Boxplots
fig, axes = plt.subplots(2, 3, figsize=(15, 8))
for ax, col in zip(axes.flat, numeric_cols):
    bp = ax.boxplot(
        df[col],
        patch_artist=True,
        boxprops=dict(facecolor="steelblue", alpha=0.7),
        medianprops=dict(color="orange", linewidth=2),
    )
    ax.set_title(f"Boxplot – {col}")
    ax.set_ylabel(col)
plt.suptitle("Q10b – Boxplots of Numeric Variables", fontsize=14)
plt.tight_layout()
plt.savefig("q10b_boxplots.png")
plt.show()

# Bar charts for categorical variables
cat_cols = [
    "inflight_meal",
    "inflight_entertainment",
    "inflight_wifi",
    "day_of_week",
    "redeye",
    "weekend",
]
fig, axes = plt.subplots(2, 3, figsize=(15, 8))
for ax, col in zip(axes.flat, cat_cols):
    counts = df[col].value_counts()
    ax.bar(counts.index, counts.values, color="darkorange", edgecolor="white")
    ax.set_title(f"Bar Chart – {col}")
    ax.set_ylabel("Count")
    ax.tick_params(axis="x", rotation=30)
plt.suptitle("Q10b – Bar Charts of Categorical Variables", fontsize=14)
plt.tight_layout()
plt.savefig("q10b_barcharts.png")
plt.show()

# ── 10c. Hypothesis Testing ────────────────────────────────────────────────
print("\n── 10c. Hypothesis Testing ──")

# One-sample z-test: Is mean coach_price significantly different from $350?
from scipy.stats import norm

mu0 = 350
z_stat = (cp.mean() - mu0) / (cp.std() / np.sqrt(len(cp)))
p_z = 2 * (1 - norm.cdf(abs(z_stat)))
print(f"\n  One-sample Z-test (H0: mean=$350):")
print(
    f"  z = {z_stat:.4f},  p = {p_z:.4f}  {'Reject H0' if p_z < 0.05 else 'Fail to reject H0'}"
)

# One-sample t-test: Is mean coach_price = $380?
t_1, p_1 = stats.ttest_1samp(cp, 380)
print(f"\n  One-sample t-test (H0: mean=$380):")
print(
    f"  t = {t_1:.4f},  p = {p_1:.4f}  {'Reject H0' if p_1 < 0.05 else 'Fail to reject H0'}"
)

# Chi-square: independence between redeye and inflight_meal
ct = pd.crosstab(df["redeye"], df["inflight_meal"])
chi2, p_chi, dof, expected = stats.chi2_contingency(ct)
print(f"\n  Chi-square test: redeye × inflight_meal")
print(
    f"  χ² = {chi2:.4f},  df = {dof},  p = {p_chi:.4f}  {'Reject H0 (dependent)' if p_chi < 0.05 else 'Fail to reject H0 (independent)'}"
)

# Chi-square: redeye × weekend
ct2 = pd.crosstab(df["redeye"], df["weekend"])
chi2b, p_chi2, dof2, _ = stats.chi2_contingency(ct2)
print(f"\n  Chi-square test: redeye × weekend")
print(
    f"  χ² = {chi2b:.4f},  df = {dof2},  p = {p_chi2:.4f}  {'Reject H0 (dependent)' if p_chi2 < 0.05 else 'Fail to reject H0 (independent)'}"
)

# ── 10d. Independent t-tests ───────────────────────────────────────────────
print("\n── 10d. Independent t-tests: Group Mean Comparisons ──")

groups = [
    ("weekend", "coach_price", "Weekend vs Weekday — Coach Price"),
    ("redeye", "coach_price", "Redeye vs Non-Redeye — Coach Price"),
    ("inflight_meal", "coach_price", "Meal vs No-Meal — Coach Price"),
]
for col, target, label in groups:
    g1 = df[df[col] == "Yes"][target]
    g2 = df[df[col] == "No"][target]
    t, p = stats.ttest_ind(g1, g2)
    print(f"\n  {label}")
    print(
        f"  Yes: ${g1.mean():.2f}  |  No: ${g2.mean():.2f}  |  t = {t:.3f}, p = {p:.4f}  {'✓ sig' if p < 0.05 else '✗ not sig'}"
    )

# ── 10e. Weekend vs Weekday Price Differences ──────────────────────────────
print("\n── 10e. Price Differences: Weekend vs Weekday ──")

wkend_coach = df[df["weekend"] == "Yes"]["coach_price"].mean()
wkday_coach = df[df["weekend"] == "No"]["coach_price"].mean()
wkend_fc = df[df["weekend"] == "Yes"]["firstclass_price"].mean()
wkday_fc = df[df["weekend"] == "No"]["firstclass_price"].mean()

print(
    f"  Coach   — Weekend: ${wkend_coach:.2f}  Weekday: ${wkday_coach:.2f}  Diff: ${wkend_coach - wkday_coach:+.2f}"
)
print(
    f"  F.Class — Weekend: ${wkend_fc:.2f}  Weekday: ${wkday_fc:.2f}  Diff: ${wkend_fc - wkday_fc:+.2f}"
)

# ── 10f. Correlation Matrix ────────────────────────────────────────────────
print("\n── 10f. Correlation Analysis ──")

cor_matrix = df[numeric_cols].corr()
print(cor_matrix.round(4).to_string())

fig, ax = plt.subplots(figsize=(8, 6))
mask = np.triu(np.ones_like(cor_matrix, dtype=bool))
sns.heatmap(
    cor_matrix, annot=True, fmt=".2f", cmap="coolwarm", mask=mask, ax=ax, linewidths=0.5
)
ax.set_title("Q10f – Correlation Heatmap")
plt.tight_layout()
plt.savefig("q10f_correlation.png")
plt.show()

# ── 10g. Linear Regression — Predict Coach Price ───────────────────────────
print("\n── 10g. Linear Regression: Predicting Coach Price ──")

# Encode categoricals
df_enc = df.copy()
for col in [
    "inflight_meal",
    "inflight_entertainment",
    "inflight_wifi",
    "redeye",
    "weekend",
]:
    df_enc[col] = (df_enc[col] == "Yes").astype(int)

X_cols = [
    "miles",
    "passengers",
    "delay",
    "hours",
    "inflight_meal",
    "inflight_entertainment",
    "inflight_wifi",
]
X = sm.add_constant(df_enc[X_cols])
y = df_enc["coach_price"]

lm = sm.OLS(y, X).fit()
print(lm.summary())

print(f"\n  Interpretation:")
print(
    f"  R² = {lm.rsquared:.4f} — the model explains {lm.rsquared * 100:.1f}% of variance in coach price."
)
print(f"  Miles and hours are the dominant predictors (positive coefficients).")

# ── 10h. Logistic Regression — Predict Redeye ─────────────────────────────
print("\n── 10h. Logistic Regression: Predicting Redeye Flights ──")

df_enc["redeye_bin"] = (df["redeye"] == "Yes").astype(int)
log_cols = ["coach_price", "miles", "hours", "passengers", "delay"]
X_log = sm.add_constant(df_enc[log_cols])
y_log = df_enc["redeye_bin"]

logit_model = sm.Logit(y_log, X_log).fit(disp=False)
print(logit_model.summary())

# Classification accuracy
pred_prob = logit_model.predict(X_log)
pred_class = (pred_prob >= 0.5).astype(int)
accuracy = (pred_class == y_log).mean()
print(f"\n  Classification Accuracy : {accuracy * 100:.2f}%")
print(f"\n  Interpretation:")
print(f"  The logistic model predicts whether a flight is a redeye.")
print(f"  Significant predictors (p<0.05) meaningfully affect the log-odds.")